# Test DuckDB

## Build
skip for duckdb

## Evaluate

### Init

In [ ]:
import sys
import os
import subprocess
import time
import tempfile
from pathlib import Path

# Add experiment and scripts directories to path
sys.path.append(os.path.abspath('.'))
sys.path.append(os.path.abspath('../scripts'))

from ExperimentRunner import DuckDBTestRunner
from extract_card_from_explain import process_data

# Project root directory
project_root = os.path.abspath('..')

# Output to experiment/checkpoint/DuckDB directory
checkpoint_dir = Path('./checkpoint/DuckDB').resolve()
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Create runner
runner = DuckDBTestRunner(project_root)

# DuckDB executable path (in running_space directory)
duckdb_exec = runner.running_space / "duckdb"

def parse_duckdb_explain_output(output_text):
    """
    Parse DuckDB EXPLAIN output, extract cardinality estimates for each query

    Parse using the process_data function from scripts/extract_card_from_explain.py

    Args:
        output_text: DuckDB EXPLAIN output text

    Returns:
        list: List containing cardinality estimates for each query (int)

    Raises:
        RuntimeError: Raised when parsing fails, contains original error info
    """
    # Use process_data function from extract_card_from_explain.py
    # This function uses regex to match "number Rows" pattern and splits data by \n\n
    try:
        cardinalities_str = process_data(output_text)
        # Convert strings to integers
        cardinalities = [int(card) for card in cardinalities_str]
        return cardinalities
    except Exception as e:
        raise RuntimeError(
            f"DuckDB EXPLAIN output parsing failed: {e}\n"
            f"Cause: DuckDB version upgrade changed the EXPLAIN output format.\n"
            f"Check if the regex in scripts/extract_card_from_explain.py needs updating."
        ) from e

def run_duckdb_explain(db_path, explain_sql_file, output_file=None):
    """
    Run DuckDB EXPLAIN queries

    Args:
        db_path: Database file path
        explain_sql_file: SQL file path containing EXPLAIN queries
        output_file: Output file path (optional, returns output text if None)

    Returns:
        str: DuckDB output text
    """
    if not duckdb_exec.exists():
        raise FileNotFoundError(f"DuckDB executable not found: {duckdb_exec}")

    if not Path(db_path).exists():
        raise FileNotFoundError(f"Database file not found: {db_path}")

    if not Path(explain_sql_file).exists():
        raise FileNotFoundError(f"SQL file not found: {explain_sql_file}")

    # Execute DuckDB using input redirection
    # Command format: duckdb db_path < explain_sql_file
    try:
        with open(explain_sql_file, 'r', encoding='utf-8') as f:
            sql_content = f.read()

        # Execute using subprocess, pass SQL via stdin
        result = subprocess.run(
            [str(duckdb_exec), str(db_path)],
            input=sql_content,
            cwd=str(runner.running_space),
            capture_output=True,
            text=True,
            timeout=3600  # 1 hour timeout
        )

        output_text = result.stdout + result.stderr

        if output_file:
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(output_text)

        if result.returncode != 0:
            raise RuntimeError(f"DuckDB execution failed, return code: {result.returncode}\n{result.stderr}")

        return output_text
    except subprocess.TimeoutExpired:
        raise RuntimeError("DuckDB execution timeout")
    except Exception as e:
        raise RuntimeError(f"Error occurred while running DuckDB: {e}")

def evaluate_duckdb_benchmark(benchmark, include_cardinalities: bool = False, preview_size: int = 5):
    """
    Evaluate DuckDB cardinality estimation on SQL queries

    Args:
        benchmark: benchmark name ('Stats', 'JOBM', 'JOBLight', 'JOBLightRanges', 'JobJoin', 'StatsJoin')
        include_cardinalities: Whether to include full cardinality list in return results
        preview_size: When not returning full cardinalities, preview first N cardinalities

    Returns:
        dict: Dictionary containing evaluation results, including:
            - total_time: Total evaluation time (seconds)
            - num_queries: Number of queries
            - output_file: Result output file path
            - cardinalities: List of cardinality estimates for each query (optional)
            - cardinality_preview: Cardinality preview (optional)
    """
    if benchmark == 'Stats':
        config = runner.get_stats_config()
        queries_file = Path(config.SUBQUERY_PATH)
        db_path = Path(config.DB_PATH)
        result_path = checkpoint_dir / "card_stats.txt"
    elif benchmark == 'JOBM':
        config = runner.get_jobm_config()
        queries_file = Path(config.SUBQUERY_PATH)
        db_path = Path(config.DB_PATH)
        result_path = checkpoint_dir / "card_jobm.txt"
    elif benchmark == 'JOBLight':
        config = runner.get_joblight_config()
        queries_file = Path(config.SUBQUERY_PATH)
        db_path = Path(config.DB_PATH)
        result_path = checkpoint_dir / "card_joblight.txt"
    elif benchmark == 'JOBLightRanges':
        config = runner.get_joblight_ranges_config()
        queries_file = Path(config.SUBQUERY_PATH)
        db_path = Path(config.DB_PATH)
        result_path = checkpoint_dir / "card_joblr.txt"
    elif benchmark == 'JobJoin':
        config = runner.get_jobjoin_config()
        queries_file = Path(config.SQL_PATH)
        db_path = Path(config.DB_PATH)
        result_path = checkpoint_dir / "card_jobjoin.txt"
    elif benchmark == 'StatsJoin':
        config = runner.get_statsjoin_config()
        queries_file = Path(config.SUBQUERY_PATH)
        db_path = Path(config.DB_PATH)
        result_path = checkpoint_dir / "card_statsjoin.txt"
    else:
        raise ValueError(f"Unsupported benchmark: {benchmark}")

    print(f"\n{'='*60}")
    print(f"Processing Benchmark: {benchmark}")
    print(f"Query file: {queries_file}")
    print(f"Database file: {db_path}")
    print(f"Result output: {result_path}")

    # 1. Generate explain.sql file
    explain_sql = runner.running_space / "explain.sql"
    runner._prepare_input_sql_with_explain(queries_file, explain_sql)
    print(f"Generated EXPLAIN SQL file: {explain_sql}")

    # 2. Run DuckDB EXPLAIN
    print(f"\nStarting DuckDB EXPLAIN...")
    start_time = time.time()

    temp_file = tempfile.NamedTemporaryFile(
        dir=str(runner.running_space),
        suffix=".txt",
        delete=False
    )
    explain_output_path = Path(temp_file.name)
    temp_file.close()

    try:
        output_text = run_duckdb_explain(
            db_path=db_path,
            explain_sql_file=explain_sql,
            output_file=str(explain_output_path)
        )

        total_time = time.time() - start_time
        print(f"DuckDB EXPLAIN completed, elapsed: {total_time:.2f}  seconds")

        # 3. Parse EXPLAIN output, extract cardinality estimates
        print(f"\nStarting to parse EXPLAIN output...")
        if explain_output_path.exists():
            with open(explain_output_path, 'r', encoding='utf-8') as f:
                output_text_from_file = f.read()
            cardinalities = parse_duckdb_explain_output(output_text_from_file)
        else:
            cardinalities = parse_duckdb_explain_output(output_text)
        num_queries = len(cardinalities)
        print(f"Parsing complete, found {num_queries} query cardinality estimates")
    finally:
        # Clean up temporary explain output file
        if explain_output_path.exists():
            explain_output_path.unlink()

    # 4. Save results to file (one cardinality per line)
    with open(result_path, 'w', encoding='utf-8') as f:
        for card in cardinalities:
            f.write(f"{card}\n")

    print(f"Results saved to: {result_path}")

    result = {
        'benchmark': benchmark,
        'total_time': total_time,
        'num_queries': num_queries,
        'output_file': str(result_path)
    }

    if include_cardinalities:
        result['cardinalities'] = cardinalities
    else:
        result['cardinality_preview'] = cardinalities[:max(preview_size, 0)]
        result['cardinality_preview_size'] = max(preview_size, 0)

    return result

print("Evaluation function defined, use evaluate_duckdb_benchmark() to evaluate each benchmark")

### Stats Benchmark

In [2]:
# Execute evaluation
stats_results = evaluate_duckdb_benchmark(benchmark='Stats')
stats_results


Processing Benchmark: Stats
Query file: /home/user/starCE/Benchmark/workloads/STATS-CEB/subquery/subquery.sql
Database file: /home/user/starCE/Benchmark/duckdb/stats.db
Result output: /home/user/starCE/experiment/checkpoint/DuckDB/card_stats.txt
[2026-06-21 13:18:12] Copied queries file to /home/user/starCE/experiment/running_space/explain.sql and added EXPLAIN
Generated EXPLAIN SQL file: /home/user/starCE/experiment/running_space/explain.sql

Starting DuckDB EXPLAIN...
DuckDB EXPLAIN completed, elapsed: 4.11 seconds

Starting to parse EXPLAIN output...
Parsing complete, found 2471 query cardinality estimates
Results saved to: /home/user/starCE/experiment/checkpoint/DuckDB/card_stats.txt


{'benchmark': 'Stats',
 'total_time': 4.1143951416015625,
 'num_queries': 2471,
 'output_file': '/home/user/starCE/experiment/checkpoint/DuckDB/card_stats.txt',
 'cardinality_preview': [40240, 6707, 6707, 6707, 201203],
 'cardinality_preview_size': 5}

### JOBM Benchmark

In [3]:
# Execute evaluation
jobm_results = evaluate_duckdb_benchmark(benchmark='JOBM')
jobm_results


Processing Benchmark: JOBM
Query file: /home/user/starCE/Benchmark/workloads/JOBM/subquery/subquery.sql
Database file: /home/user/starCE/Benchmark/duckdb/imdb.db
Result output: /home/user/starCE/experiment/checkpoint/DuckDB/card_jobm.txt
[2026-06-21 13:18:17] Copied queries file to /home/user/starCE/experiment/running_space/explain.sql and added EXPLAIN
Generated EXPLAIN SQL file: /home/user/starCE/experiment/running_space/explain.sql

Starting DuckDB EXPLAIN...
DuckDB EXPLAIN completed, elapsed: 18.64 seconds

Starting to parse EXPLAIN output...
Parsing complete, found 6424 query cardinality estimates
Results saved to: /home/user/starCE/experiment/checkpoint/DuckDB/card_jobm.txt


{'benchmark': 'JOBM',
 'total_time': 18.637667179107666,
 'num_queries': 6424,
 'output_file': '/home/user/starCE/experiment/checkpoint/DuckDB/card_jobm.txt',
 'cardinality_preview': [0, 300, 7521, 7521, 0],
 'cardinality_preview_size': 5}

### JOBLight Benchmark

In [4]:
# Execute evaluation
joblight_results = evaluate_duckdb_benchmark(benchmark='JOBLight')
joblight_results


Processing Benchmark: JOBLight
Query file: /home/user/starCE/Benchmark/workloads/JOBLight/subquery/subquery.sql
Database file: /home/user/starCE/Benchmark/duckdb/imdb.db
Result output: /home/user/starCE/experiment/checkpoint/DuckDB/card_joblight.txt
[2026-06-21 13:18:36] Copied queries file to /home/user/starCE/experiment/running_space/explain.sql and added EXPLAIN
Generated EXPLAIN SQL file: /home/user/starCE/experiment/running_space/explain.sql

Starting DuckDB EXPLAIN...
DuckDB EXPLAIN completed, elapsed: 0.49 seconds

Starting to parse EXPLAIN output...
Parsing complete, found 451 query cardinality estimates
Results saved to: /home/user/starCE/experiment/checkpoint/DuckDB/card_joblight.txt


{'benchmark': 'JOBLight',
 'total_time': 0.48886680603027344,
 'num_queries': 451,
 'output_file': '/home/user/starCE/experiment/checkpoint/DuckDB/card_joblight.txt',
 'cardinality_preview': [3912808, 3912808, 3912808, 3912808, 46953704],
 'cardinality_preview_size': 5}

### JOBLightRanges Benchmark

In [5]:
# Execute evaluation
joblr_results = evaluate_duckdb_benchmark(benchmark='JOBLightRanges')
joblr_results


Processing Benchmark: JOBLightRanges
Query file: /home/user/starCE/Benchmark/workloads/JOBLightRanges/subquery/subquery.sql
Database file: /home/user/starCE/Benchmark/duckdb/imdb.db
Result output: /home/user/starCE/experiment/checkpoint/DuckDB/card_joblr.txt
[2026-06-21 13:18:36] Copied queries file to /home/user/starCE/experiment/running_space/explain.sql and added EXPLAIN
Generated EXPLAIN SQL file: /home/user/starCE/experiment/running_space/explain.sql

Starting DuckDB EXPLAIN...
DuckDB EXPLAIN completed, elapsed: 8.87 seconds

Starting to parse EXPLAIN output...
Parsing complete, found 8292 query cardinality estimates
Results saved to: /home/user/starCE/experiment/checkpoint/DuckDB/card_joblr.txt


{'benchmark': 'JOBLightRanges',
 'total_time': 8.867777347564697,
 'num_queries': 8292,
 'output_file': '/home/user/starCE/experiment/checkpoint/DuckDB/card_joblr.txt',
 'cardinality_preview': [1956405, 4695371, 1956405, 3912808, 1956405],
 'cardinality_preview_size': 5}

### JobJoin Benchmark

In [6]:
# Execute evaluation
jobjoin_results = evaluate_duckdb_benchmark(benchmark='JobJoin')
jobjoin_results


Processing Benchmark: JobJoin
Query file: /home/user/project/starCE/Benchmark/workloads/JobJoin/subquery/subquery.sql
Database file: /home/user/project/starCE/Benchmark/duckdb/imdb.db
Result output: /home/user/project/starCE/experiment/checkpoint/DuckDB/card_jobjoin.txt
[2026-06-21 21:45:58] Copied queries file to /home/user/project/starCE/experiment/running_space/explain.sql and added EXPLAIN
Generated EXPLAIN SQL file: /home/user/project/starCE/experiment/running_space/explain.sql

Starting DuckDB EXPLAIN...
DuckDB EXPLAIN completed, elapsed: 12.77 seconds

Starting to parse EXPLAIN output...
Parsing complete, found 9565 query cardinality estimates
Results saved to: /home/user/project/starCE/experiment/checkpoint/DuckDB/card_jobjoin.txt


{'benchmark': 'JobJoin',
 'total_time': 12.772614240646362,
 'num_queries': 9565,
 'output_file': '/home/user/project/starCE/experiment/checkpoint/DuckDB/card_jobjoin.txt',
 'cardinality_preview': [2609129, 3378693, 3590825, 1787077, 1380035],
 'cardinality_preview_size': 5}

### StatsJoin Benchmark

In [ ]:
# Execute evaluation
statsjoin_results = evaluate_duckdb_benchmark(benchmark='StatsJoin')
statsjoin_results